# CofC Soccer — COUGS Data Pipeline
### Week 1–2: From Raw XML to Clean CSVs
**College of Charleston | Soccer Analytics**

---

This notebook walks you through the first stage of the CofC analytics pipeline:
taking raw XML files exported from Wyscout and Spiideo and turning them into
clean, structured CSV files that can be analyzed and eventually loaded into a database.

**By the end of this notebook you will:**
- Understand what the raw source files look like and where they come from
- Know how the pipeline parses each file type
- Have clean CSV outputs for every match in your folder

**You are not writing to any database yet.** That comes later.
All outputs land in your Google Drive as CSV files.

---

> 💡 **How to use this notebook**
> Run each cell in order using **Shift + Enter**.
> Read the markdown explanations — they tell you *why* each step exists, not just what to run.
> If a cell fails, check the ❌ message before moving on.


---
## Section 1 — Setup
Run these once at the start of every session.


In [ ]:
# Mount your Google Drive — this is where your files live
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")

In [ ]:
# Install the one library Colab doesn't have by default
!pip install python-dotenv --quiet
print("✅ Dependencies ready")

In [ ]:
# ── EDIT THIS to match your Drive folder location ──────────────
PROJECT_ROOT = "/content/drive/MyDrive/CofC_Pipeline"
# ──────────────────────────────────────────────────────────────

import sys, os
from pathlib import Path

# Point Python to the pipeline source code
src_path = os.path.join(PROJECT_ROOT, "pipeline", "src")
sys.path.insert(0, src_path)

# Check that the key folders and files exist
required = [
    "pipeline/src/parse_wyscout.py",
    "pipeline/src/parse_spiideo.py",
    "pipeline/src/attribute.py",
    "pipeline/src/manifest.py",
    "matches/matches_manifest.csv",
]

print("Checking project structure...")
all_ok = True
for p in required:
    full = os.path.join(PROJECT_ROOT, p)
    exists = os.path.exists(full)
    print(f"  {'✅' if exists else '❌ MISSING'}  {p}")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("✅ Everything looks good — ready to go")
else:
    print("❌ Fix missing files before continuing")
    print("   Ask if you're not sure where these should be")

In [ ]:
# Import the pipeline modules
# These are Python files in pipeline/src/ — open them and read them!
# They do the actual work of parsing the XML files.

from parse_wyscout import parse_sportscode, parse_effective_time, estimate_minutes_played
from parse_spiideo import parse_spiideo
from attribute import calculate_offset, attribute_players
from manifest import load_manifest

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ Pipeline modules imported")
print()
print("Modules loaded:")
print("  parse_wyscout  — reads Wyscout XML files")
print("  parse_spiideo  — reads Spiideo tag XML files")
print("  attribute      — matches Spiideo events to Wyscout players")
print("  manifest       — reads the matches_manifest.csv")

---
## Section 2 — Understand the Source Files

Before running any pipeline code, let's look at the raw files.
Understanding the inputs makes everything else make sense.

### Where the files come from
- **Wyscout** — the video analysis platform CofC uses. After each match, staff
  export XML files that describe every tagged event in the video.
- **Spiideo** — a separate tagging tool where coaches tag COUG moments
  (ASET = defensive actions, PEAK = attacking actions) during video review.

### File naming convention
Every file follows this pattern:
```
matches/
└── 2025/
    └── 2025-11-02_uncw/           ← folder: YYYY-MM-DD_opponent
        ├── 2025-11-02_uncw_cfc_sportscode.xml    ← Wyscout (primary)
        ├── 2025-11-02_uncw_cfc_effective_time.xml← Wyscout effective time
        └── 2025-11-02_uncw_spiideo.xml           ← Spiideo COUG tags
```
The folder name must exactly match a row in `matches_manifest.csv`.


In [ ]:
# Load the manifest — the master list of all matches
MANIFEST_PATH = Path(PROJECT_ROOT) / "matches" / "matches_manifest.csv"
MATCHES_DIR   = Path(PROJECT_ROOT) / "matches"

manifest = load_manifest(MANIFEST_PATH)

# Display it as a readable table
df_manifest = pd.read_csv(MANIFEST_PATH)
print(f"Manifest loaded — {len(manifest)} matches\n")
display(df_manifest)

In [ ]:
# Check which XML files are present for each match
# This tells you which pipeline mode each match can run in:
#   Full    = Wyscout + Spiideo (best — full player attribution)
#   Wyscout = Wyscout only (player data, no COUG scores yet)
#   Spiideo = Spiideo only (COUG moments, no player names yet)

print("Match folder status:")
print("-" * 65)

for slug, meta in sorted(manifest.items()):
    match_dir = MATCHES_DIR / meta.season / slug
    has_wyscout = (match_dir / f"{slug}_cfc_sportscode.xml").exists()
    has_spiideo = (match_dir / f"{slug}_spiideo.xml").exists()

    if has_wyscout and has_spiideo:
        status = "🟢 Full pipeline"
    elif has_wyscout:
        status = "🟡 Wyscout only"
    elif has_spiideo:
        status = "🟡 Spiideo only"
    else:
        status = "🔴 No XML files found"

    print(f"  {slug:<38} {status}")

print()
print("🟢 Full = COUG scores with player names")
print("🟡 Partial = some data, will improve when both files exist")
print("🔴 Check file naming or that files were exported from Wyscout/Spiideo")

### Peek inside a raw XML file
Run the cell below to see what the pipeline is actually reading.
This is the raw Wyscout Sportscode XML — notice how it's structured
around `<instance>` elements, each representing one tagged event.


In [ ]:
# Peek at the raw XML for one match
# Change MATCH_SLUG to any slug from the manifest above

MATCH_SLUG = "2025-11-02_uncw"   # ← EDIT THIS

match_dir = MATCHES_DIR / "2025" / MATCH_SLUG
sportscode_file = match_dir / f"{MATCH_SLUG}_cfc_sportscode.xml"

if sportscode_file.exists():
    # Read first 3000 characters of the raw XML
    raw = sportscode_file.read_bytes()
    # Try UTF-16 first (Wyscout default), then UTF-8
    for enc in ("utf-16", "utf-8"):
        try:
            text = raw.decode(enc)
            break
        except:
            continue

    print(f"File: {sportscode_file.name}")
    print(f"Size: {sportscode_file.stat().st_size / 1024:.1f} KB")
    print()
    print("First 3000 characters of raw XML:")
    print("=" * 60)
    print(text[:3000])
    print("=" * 60)
    print()
    print("Notice:")
    print("  - Each <instance> is one tagged event")
    print("  - <code> tells us who or what was tagged")
    print("  - <start> and <end> are timestamps in seconds")
    print("  - <text> labels describe the event type and outcome")
else:
    print(f"❌ File not found: {sportscode_file}")
    print("   Check MATCH_SLUG and that the file exists in the right folder")

---
## Section 3 — Parse a Single Match

Now we run the parser on that same file and watch the messy XML
become a clean, readable dataframe.

This is what the pipeline code in `parse_wyscout.py` does — it reads every
`<instance>` element and extracts the fields we care about into a Python list,
which we then put into a pandas DataFrame.


In [ ]:
# Parse the Wyscout Sportscode XML
# This is the richest file — it has player events, team events, and half markers

print(f"Parsing: {MATCH_SLUG}")
print("=" * 60)

wyscout_data = parse_sportscode(sportscode_file)

print(f"\nWhat came back:")
print(f"  player_events : {len(wyscout_data['player_events'])} rows")
print(f"  team_events   : {len(wyscout_data['team_events'])} rows")
print(f"  halves        : {wyscout_data['halves']}")

In [ ]:
# Turn player events into a DataFrame so we can read them
df_players = pd.DataFrame(wyscout_data["player_events"])

print("Player events — first 10 rows:")
print("(Each row is one tagged action by a player in the match)")
print()
display(df_players[["jersey", "name", "start", "end", "outcome", "labels"]].head(10))

print(f"\nUnique players: {df_players['name'].nunique()}")
print(f"Unique event types (from labels):")

# Flatten all labels to see what types of events were tagged
all_labels = [l for labels in df_players["labels"] for l in labels]
from collections import Counter
for label, count in Counter(all_labels).most_common(15):
    print(f"  {label:<35} {count}")

In [ ]:
# Parse the Spiideo XML (if it exists for this match)
spiideo_file = match_dir / f"{MATCH_SLUG}_spiideo.xml"

if spiideo_file.exists():
    print(f"Parsing Spiideo XML...")
    spiideo_data = parse_spiideo(spiideo_file)

    df_spiideo = pd.DataFrame(spiideo_data["coug_events"])
    print(f"\nCOUG events — first 10 rows:")
    print("(These are the moments coaches tagged as ASET or PEAK during video review)")
    print()
    display(df_spiideo.head(10))

    print(f"\nASET subtypes (defensive moments):")
    aset = df_spiideo[df_spiideo["category"] == "ASET"]["subtype"].value_counts()
    print(aset.to_string())

    print(f"\nPEAK subtypes (attacking moments):")
    peak = df_spiideo[df_spiideo["category"] == "PEAK"]["subtype"].value_counts()
    print(peak.to_string())
else:
    print(f"No Spiideo file for {MATCH_SLUG} — only Wyscout data available")
    spiideo_data = None
    df_spiideo = None

### The attribution step
When both files exist, we can link Spiideo COUG moments to specific players.

How? Spiideo timestamps tell us *when* a COUG moment happened.
Wyscout timestamps tell us *which player* was active at that moment.
We match them within a ±15 second window and score the best candidate.

This is the key insight of the whole pipeline — two separate systems,
aligned by time.


In [ ]:
# Run attribution — match Spiideo COUG events to Wyscout players
# Only runs if we have both files

if spiideo_data:
    print("Calculating timestamp offset between Wyscout and Spiideo...")
    offset = calculate_offset(
        wyscout_halves=wyscout_data["halves"],
        spiideo_all_events=spiideo_data["all_events"],
        spiideo_recording_start=None,
        manual_offset=None,
    )

    print(f"\nOffset: {offset:.1f} seconds")
    print("(This aligns the two systems' clocks — Spiideo starts recording")
    print(" before kickoff, so its timestamps are larger than Wyscout's)")

    print("\nAttributing players to COUG events...")
    first_half_start = wyscout_data["halves"].get("first_start", 2.0)
    attributed = attribute_players(
        coug_events=spiideo_data["coug_events"],
        player_events=wyscout_data["player_events"],
        offset=offset,
        first_half_start=first_half_start,
    )

    # Preview
    df_attributed = pd.DataFrame([{
        "minute":   round(ev.get("match_minute", 0), 1),
        "category": ev["category"],
        "subtype":  ev["subtype"],
        "player":   ev["player"]["name"] if ev.get("player") else "⚠️ unattributed",
        "outcome":  ev["player"]["outcome"] if ev.get("player") else "",
        "score":    ev.get("attribution_score", 0),
    } for ev in attributed])

    attributed_pct = (df_attributed["player"] != "⚠️ unattributed").mean() * 100
    print(f"\nAttribution rate: {attributed_pct:.0f}%")
    print()
    display(df_attributed.head(20))

    if attributed_pct < 50:
        print()
        print("⚠️  Attribution rate is low.")
        print("   This usually means the offset is off.")
        print("   Note the MANUAL_OFFSET override in Section 4 — ask your supervisor.")
else:
    print("Skipping attribution — no Spiideo file for this match")
    attributed = []

---
## Section 4 — Export to CSV

Now we save everything to well-named CSV files in your Drive.

These CSVs are the deliverable for this stage of the pipeline.
They're human-readable, shareable, and ready to load into a database
when we build the schema later.

### Output file naming convention
```
outputs/
└── 2025/
    └── 2025-11-02_uncw/
        ├── 2025-11-02_uncw_players.csv        ← all player events from Wyscout
        ├── 2025-11-02_uncw_coug_events.csv    ← COUG moments from Spiideo
        ├── 2025-11-02_uncw_attributed.csv     ← COUG events with player names
        └── 2025-11-02_uncw_minutes.csv        ← estimated minutes per player
```


In [ ]:
# Set up the output directory for this match
OUTPUT_DIR = Path(PROJECT_ROOT) / "outputs" / "2025" / MATCH_SLUG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR}")

In [ ]:
# Export 1: Player events
df_players_out = pd.DataFrame(wyscout_data["player_events"])

# Flatten labels list to a string for CSV compatibility
df_players_out["labels"] = df_players_out["labels"].apply(lambda x: " | ".join(x))

out_path = OUTPUT_DIR / f"{MATCH_SLUG}_players.csv"
df_players_out.to_csv(out_path, index=False)
print(f"✅ Player events saved: {out_path.name}")
print(f"   {len(df_players_out)} rows, {df_players_out['name'].nunique()} unique players")

In [ ]:
# Export 2: Spiideo COUG events (if available)
if spiideo_data:
    df_spiideo_out = pd.DataFrame(spiideo_data["coug_events"])
    out_path = OUTPUT_DIR / f"{MATCH_SLUG}_coug_events.csv"
    df_spiideo_out.to_csv(out_path, index=False)
    print(f"✅ COUG events saved: {out_path.name}")
    print(f"   {len(df_spiideo_out)} rows ({df_spiideo_out['category'].value_counts().to_dict()})")
else:
    print("⏭️  No Spiideo file — skipping COUG events export")

In [ ]:
# Export 3: Attributed events — the main output
if attributed:
    df_attr_out = pd.DataFrame([{
        "match":             MATCH_SLUG,
        "minute":            round(ev.get("match_minute", 0), 1),
        "category":          ev["category"],
        "subtype":           ev["subtype"],
        "spiideo_t":         ev["spiideo_t"],
        "wyscout_t":         ev.get("wyscout_t"),
        "player_name":       ev["player"]["name"] if ev.get("player") else None,
        "player_jersey":     ev["player"]["jersey"] if ev.get("player") else None,
        "outcome":           ev["player"]["outcome"] if ev.get("player") else None,
        "attribution_score": ev.get("attribution_score", 0),
        "spiideo_code":      ev["spiideo_code"],
    } for ev in attributed])

    out_path = OUTPUT_DIR / f"{MATCH_SLUG}_attributed.csv"
    df_attr_out.to_csv(out_path, index=False)
    print(f"✅ Attributed events saved: {out_path.name}")
    print(f"   {len(df_attr_out)} rows | {attributed_pct:.0f}% attributed to a player")
else:
    print("⏭️  No attribution data — skipping (need both Wyscout + Spiideo)")

In [ ]:
# Export 4: Minutes played per player
minutes_lookup = estimate_minutes_played(wyscout_data["player_events"])
df_minutes = pd.DataFrame([
    {"player_name": name, "estimated_minutes": mins}
    for name, mins in sorted(minutes_lookup.items())
])

out_path = OUTPUT_DIR / f"{MATCH_SLUG}_minutes.csv"
df_minutes.to_csv(out_path, index=False)
print(f"✅ Minutes saved: {out_path.name}")
print()
display(df_minutes)
print()
print("Note: minutes are estimated from first→last event timestamp.")
print("This is an approximation — exact substitution data comes later.")

In [ ]:
# Summary of what was exported
print(f"Match: {MATCH_SLUG}")
print(f"Output folder: {OUTPUT_DIR}")
print()
print("Files created:")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    rows = sum(1 for _ in open(f)) - 1  # subtract header
    print(f"  {f.name:<55} {rows} rows")

---
## Section 5 — Batch All Matches

Once a single match works, run this section to process every match
in the manifest. Same logic, looped.

Each match gets its own output folder. Matches with missing files are
skipped gracefully with a note — not treated as errors.


In [ ]:
# Batch process all matches in the manifest
SEASON = "2025"
MANUAL_OFFSET = None   # set to an integer to override offset for ALL matches
                       # (usually leave as None — each match calculates its own)

results = {"success": [], "skipped": [], "failed": []}

print(f"Processing all {SEASON} matches...")
print("=" * 65)

for slug, meta in sorted(manifest.items()):
    match_dir    = MATCHES_DIR / meta.season / slug
    output_dir   = Path(PROJECT_ROOT) / "outputs" / meta.season / slug
    output_dir.mkdir(parents=True, exist_ok=True)

    has_wyscout = (match_dir / f"{slug}_cfc_sportscode.xml").exists()
    has_spiideo = (match_dir / f"{slug}_spiideo.xml").exists()

    if not has_wyscout and not has_spiideo:
        print(f"\n⏭️  {slug} — no XML files, skipping")
        results["skipped"].append(slug)
        continue

    print(f"\n📁 {slug}")

    try:
        wyscout_data = spiideo_data = attributed = None

        # Parse Wyscout
        if has_wyscout:
            sportscode_file = match_dir / f"{slug}_cfc_sportscode.xml"
            wyscout_data = parse_sportscode(sportscode_file)

        # Parse Spiideo
        if has_spiideo:
            spiideo_file = match_dir / f"{slug}_spiideo.xml"
            spiideo_data = parse_spiideo(spiideo_file)

        # Attribute
        if wyscout_data and spiideo_data:
            offset = calculate_offset(
                wyscout_halves=wyscout_data["halves"],
                spiideo_all_events=spiideo_data["all_events"],
                spiideo_recording_start=meta.spiideo_recording_start,
                manual_offset=MANUAL_OFFSET,
            )
            first_half_start = wyscout_data["halves"].get("first_start", 2.0)
            attributed = attribute_players(
                coug_events=spiideo_data["coug_events"],
                player_events=wyscout_data["player_events"],
                offset=offset,
                first_half_start=first_half_start,
            )

        # Export CSVs
        files_written = []

        if wyscout_data:
            df_p = pd.DataFrame(wyscout_data["player_events"])
            df_p["labels"] = df_p["labels"].apply(lambda x: " | ".join(x))
            p = output_dir / f"{slug}_players.csv"
            df_p.to_csv(p, index=False)
            files_written.append(f"players ({len(df_p)} rows)")

            mins = estimate_minutes_played(wyscout_data["player_events"])
            df_m = pd.DataFrame([{"player_name": k, "estimated_minutes": v} for k,v in mins.items()])
            df_m.to_csv(output_dir / f"{slug}_minutes.csv", index=False)
            files_written.append(f"minutes ({len(df_m)} players)")

        if spiideo_data:
            df_s = pd.DataFrame(spiideo_data["coug_events"])
            df_s.to_csv(output_dir / f"{slug}_coug_events.csv", index=False)
            files_written.append(f"coug_events ({len(df_s)} rows)")

        if attributed:
            df_a = pd.DataFrame([{
                "match":             slug,
                "minute":            round(ev.get("match_minute", 0), 1),
                "category":          ev["category"],
                "subtype":           ev["subtype"],
                "spiideo_t":         ev["spiideo_t"],
                "wyscout_t":         ev.get("wyscout_t"),
                "player_name":       ev["player"]["name"] if ev.get("player") else None,
                "player_jersey":     ev["player"]["jersey"] if ev.get("player") else None,
                "outcome":           ev["player"]["outcome"] if ev.get("player") else None,
                "attribution_score": ev.get("attribution_score", 0),
                "spiideo_code":      ev["spiideo_code"],
            } for ev in attributed])
            df_a.to_csv(output_dir / f"{slug}_attributed.csv", index=False)
            attr_pct = (df_a["player_name"].notna()).mean() * 100
            files_written.append(f"attributed ({len(df_a)} events, {attr_pct:.0f}% matched)")

        for f in files_written:
            print(f"    ✅ {f}")

        results["success"].append(slug)

    except Exception as e:
        print(f"    ❌ FAILED: {e}")
        results["failed"].append(slug)

# Summary
print(f"\n{'='*65}")
print(f"BATCH COMPLETE")
print(f"  ✅ Success:  {len(results['success'])}")
print(f"  ⏭️  Skipped: {len(results['skipped'])}")
print(f"  ❌ Failed:   {len(results['failed'])}")
if results["failed"]:
    print(f"\n  Failed — run Section 3 individually to debug:")
    for f in results["failed"]:
        print(f"    - {f}")
print(f"{'='*65}")

---
## Section 6 — Build the Season COUG Table

Now that we have attributed CSVs for every match, we can stack them
into a season-level view and compute each player's running COUG totals.

This is the first time we see the full picture — every player,
every match, one table.


In [ ]:
# Load all attributed CSVs and combine into one season DataFrame
from pathlib import Path

output_base = Path(PROJECT_ROOT) / "outputs" / SEASON

all_attributed = []
for match_dir in sorted(output_base.iterdir()):
    slug = match_dir.name
    attr_file = match_dir / f"{slug}_attributed.csv"
    if attr_file.exists():
        df = pd.read_csv(attr_file)
        all_attributed.append(df)

if not all_attributed:
    print("No attributed CSVs found yet.")
    print("Run Section 5 first, or check that you have Spiideo + Wyscout files.")
else:
    df_season = pd.concat(all_attributed, ignore_index=True)
    print(f"Season data loaded:")
    print(f"  Matches:       {df_season['match'].nunique()}")
    print(f"  Total events:  {len(df_season)}")
    print(f"  Players seen:  {df_season['player_name'].nunique()}")
    print(f"  Attribution:   {df_season['player_name'].notna().mean()*100:.0f}%")

In [ ]:
# Compute season COUG totals per player
# ASET = defensive moments, PEAK = attacking moments

if all_attributed:
    # Separate ASET and PEAK
    df_aset = df_season[df_season["category"] == "ASET"].groupby("player_name").size().rename("aset_events")
    df_peak = df_season[df_season["category"] == "PEAK"].groupby("player_name").size().rename("peak_events")
    df_matches = df_season.dropna(subset=["player_name"]).groupby("player_name")["match"].nunique().rename("matches")

    season_coug = pd.concat([df_aset, df_peak, df_matches], axis=1).fillna(0).reset_index()
    season_coug["total_events"] = season_coug["aset_events"] + season_coug["peak_events"]
    season_coug = season_coug.sort_values("total_events", ascending=False)

    print("Season COUG Table:")
    print()
    display(season_coug)

    # Save it
    out_path = Path(PROJECT_ROOT) / "outputs" / f"season_{SEASON}_coug_table.csv"
    season_coug.to_csv(out_path, index=False)
    print(f"\n✅ Saved: {out_path.name}")

In [ ]:
# Visualize the season COUG table
import matplotlib.pyplot as plt

if all_attributed and not season_coug.empty:
    COFC_COLOR = "#6B0032"

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Stacked ASET / PEAK bar chart
    ax1 = axes[0]
    top = season_coug.head(15).sort_values("total_events", ascending=True)
    ax1.barh(top["player_name"], top["aset_events"], color="#2ecc71", alpha=0.85, label="ASET (Defense)")
    ax1.barh(top["player_name"], top["peak_events"],
             left=top["aset_events"], color="#e74c3c", alpha=0.85, label="PEAK (Attack)")
    ax1.set_title("Season COUG Table — Top 15 Players", fontsize=13, fontweight="bold")
    ax1.set_xlabel("COUG Events")
    ax1.legend()

    # Appearances vs total events scatter
    ax2 = axes[1]
    ax2.scatter(season_coug["matches"], season_coug["total_events"],
                color=COFC_COLOR, s=100, alpha=0.8, zorder=3)
    for _, row in season_coug.iterrows():
        label = str(row["player_name"]).split(".")[-1].strip()
        ax2.annotate(label, (row["matches"], row["total_events"]),
                     textcoords="offset points", xytext=(5, 3), fontsize=8)
    ax2.set_title("Appearances vs COUG Events", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Matches in Dataset")
    ax2.set_ylabel("Total COUG Events")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()

    fig_dir = Path(PROJECT_ROOT) / "outputs" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    fig_path = fig_dir / f"season_{SEASON}_coug_table.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {fig_path.name}")

---
## Section 7 — What Comes Next

You've now completed Stage 1 of the pipeline:

```
Raw XML files  →  Parsed DataFrames  →  Clean CSVs  →  Season COUG Table
```

**What these CSVs will be used for:**
- Loaded into a Supabase database (Stage 2 — schema design TBD)
- Used to build the coaching dashboard
- Foundation for the player similarity and evaluation models

**Your job as an undergrad operator:**
- Add new match folders as files become available from Wyscout/Spiideo
- Run Section 4 (single match) to verify, then Section 5 (batch) to update all
- Flag any match with attribution rate < 50% — it may need a manual offset
- Keep `matches_manifest.csv` up to date with scores and metadata

---

### Troubleshooting reference

| Problem | Likely cause | Fix |
|---|---|---|
| `FileNotFoundError` | File named wrong | Check Section 2 status checker |
| Match not in manifest | Missing CSV row | Add row to `matches_manifest.csv` |
| Attribution < 50% | Offset is wrong | Set `MANUAL_OFFSET` in Section 3/5 |
| No COUG events | No Spiideo file yet | Normal — wait for Spiideo export |
| Empty season table | No attributed files | Run Section 5 first |

---
*CofC Soccer Analytics Pipeline | Stage 1 — CSV Export*
*Version 1.0 | Built for Google Colab*
